In [1]:
!pip install -q transformers datasets accelerate scikit-learn pandas numpy joblib

In [2]:
import os
import numpy as np
import pandas as pd
import torch
import joblib

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, jaccard_score

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

In [3]:
import pandas as pd

data_path = "/kaggle/input/datasets/overgame1234/new-data-csv/new_data.csv"
df = pd.read_csv(data_path)

In [4]:
df = df[["text", "tags"]].copy()
df = df.dropna(subset=["text", "tags"])

df["text"] = df["text"].astype(str).str.strip()
df["tags"] = df["tags"].astype(str).str.strip()

df = df[(df["text"] != "") & (df["tags"] != "")]



In [5]:
df["label_list"] = df["tags"].apply(lambda x: x.split())
df[["text", "tags", "label_list"]].head()  


,text,tags,label_list
0,loop through elements of list in a pandas data...,python pandas,"[python, pandas]"
1,ssl error when running pip search in python 2....,python ssl,"[python, ssl]"
2,how do i clear errno in c# how do i clear errn...,c# linux,"[c#, linux]"
3,segmentation fault as soon the binary launch h...,linux debugging,"[linux, debugging]"
4,changing data in jsp using ajax i have jsp pag...,javascript java,"[javascript, java]"


In [6]:
df["num_labels"] = df["label_list"].apply(len)

print(df["num_labels"].value_counts().sort_index())
df[["text", "tags", "label_list", "num_labels"]].head()

num_labels
1    1000000
2     701000
3     112707
4       7949
5        304
Name: count, dtype: int64


,text,tags,label_list,num_labels
0,loop through elements of list in a pandas data...,python pandas,"[python, pandas]",2
1,ssl error when running pip search in python 2....,python ssl,"[python, ssl]",2
2,how do i clear errno in c# how do i clear errn...,c# linux,"[c#, linux]",2
3,segmentation fault as soon the binary launch h...,linux debugging,"[linux, debugging]",2
4,changing data in jsp using ajax i have jsp pag...,javascript java,"[javascript, java]",2


In [7]:
df_1 = df[df["num_labels"] == 1].copy()
df_2 = df[df["num_labels"] == 2].copy()
df_3_plus = df[df["num_labels"] >= 3].copy()



In [8]:
RANDOM_STATE = 42

df_1_sample = df_1.sample(
    n=min(900000, len(df_1)),
    random_state=RANDOM_STATE
).copy()

df_2_sample = df_2.sample(
    n=min(360000, len(df_2)),
    random_state=RANDOM_STATE
).copy()

df_final = pd.concat([df_1_sample, df_2_sample, df_3_plus], ignore_index=True)
df_final = df_final.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print("Final shape:", df_final.shape)
print(df_final["num_labels"].value_counts().sort_index())

Final shape: (1380960, 4)
num_labels
1    900000
2    360000
3    112707
4      7949
5       304
Name: count, dtype: int64


In [9]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df_final["label_list"])

print("Number of labels:", len(mlb.classes_))
print("y shape:", y.shape)
print("First labels:", mlb.classes_[:20])

Number of labels: 52
y shape: (1380960, 52)
First labels: ['apache' 'asp.net-core' 'authentication' 'azure' 'bash' 'c#'
 'computer-vision' 'cors' 'cuda' 'debugging' 'deep-learning' 'django'
 'dns' 'docker' 'fastapi' 'firewall' 'gpu' 'http' 'inference' 'java']


In [10]:
X = df_final["text"].tolist()



In [11]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42,
    shuffle=True
)



In [12]:
from datasets import Dataset
import numpy as np

y_train = y_train.astype(np.float32)
y_val = y_val.astype(np.float32)
y_test = y_test.astype(np.float32)

train_dataset = Dataset.from_dict({
    "text": X_train if isinstance(X_train, list) else X_train.tolist(),
    "labels": y_train.tolist()
})

val_dataset = Dataset.from_dict({
    "text": X_val if isinstance(X_val, list) else X_val.tolist(),
    "labels": y_val.tolist()
})

test_dataset = Dataset.from_dict({
    "text": X_test if isinstance(X_test, list) else X_test.tolist(),
    "labels": y_test.tolist()
})

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 1104768
Val: 138096
Test: 138096


In [13]:
model_name = "microsoft/unixcoder-base-nine"
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

In [14]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/1104768 [00:00<?, ? examples/s]

Map:   0%|          | 0/138096 [00:00<?, ? examples/s]

Map:   0%|          | 0/138096 [00:00<?, ? examples/s]

In [15]:
num_labels = y_train.shape[1]

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

print("Num labels:", num_labels)

pytorch_model.bin:   0%|          | 0.00/504M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: microsoft/unixcoder-base-nine
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
embeddings.position_ids    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Num labels: 52


In [16]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/unixcode_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=28,
    per_device_eval_batch_size=28,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_at_3",
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

model.safetensors:   0%|          | 0.00/504M [00:00<?, ?B/s]

In [17]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

def top_k_binary_predictions(y_scores, k):
    y_pred = np.zeros_like(y_scores, dtype=int)
    topk_idx = np.argsort(-y_scores, axis=1)[:, :k]

    for i in range(y_scores.shape[0]):
        y_pred[i, topk_idx[i]] = 1

    return y_pred

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    results = {}

    for k in [1, 2, 3, 4,5]:
        preds_k = top_k_binary_predictions(logits, k)

        results[f"precision_at_{k}"] = precision_score(labels, preds_k, average="micro", zero_division=0)
        results[f"recall_at_{k}"] = recall_score(labels, preds_k, average="micro", zero_division=0)
        results[f"f1_at_{k}"] = f1_score(labels, preds_k, average="micro", zero_division=0)

    return results

In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Precision At 1,Recall At 1,F1 At 1,Precision At 2,Recall At 2,F1 At 2,Precision At 3,Recall At 3,F1 At 3,Precision At 4,Recall At 4,F1 At 4,Precision At 5,Recall At 5,F1 At 5
1,0.055561,0.042229,0.929129,0.644193,0.760859,0.634569,0.879932,0.737375,0.456043,0.948564,0.615953,0.350425,0.971839,0.515111,0.283086,0.981358,0.439416
2,0.040486,0.040336,0.934712,0.648064,0.765431,0.637274,0.883682,0.740518,0.457573,0.951747,0.618020,0.351279,0.974209,0.516367,0.283620,0.983211,0.440246


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [ ]:
val_results = trainer.evaluate(eval_dataset=val_dataset)
print("Validation Results:", val_results)

test_results = trainer.evaluate(eval_dataset=test_dataset)
print("Test Results:", test_results)